# 02. EDA Time Series

В этом ноутбуке изучаем динамику продаж, сезонные паттерны, промо, праздники и внешние факторы.

## 1. Load processed data

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.visualization import plot_time_series, save_plot

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
IMAGES = PROJECT_ROOT / "images"
IMAGES.mkdir(parents=True, exist_ok=True)

processed_path = DATA_PROCESSED / "daily_sales_series.csv"
if not processed_path.exists():
    raise FileNotFoundError("Run notebooks/01_data_preparation.ipynb first.")

df = pd.read_csv(processed_path, parse_dates=["date"])
print(df.shape)
display(df.head())

## 2. Sales trend

In [ ]:
plot_time_series(df, target_col="sales_clean", title="Daily Sales Trend")
save_plot(IMAGES / "sales_trend.png")
plt.show()

## 3. Distribution of sales

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df["sales_clean"], bins=40)
ax.set_title("Distribution of Clean Sales")
ax.set_xlabel("sales_clean")
ax.set_ylabel("frequency")
ax.grid(True)
plt.show()

display(df["sales_clean"].describe())

## 4. Monthly sales

In [ ]:
monthly_sales = (
    df.assign(month_period=df["date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month_period", as_index=False)
    .agg(sales_clean=("sales_clean", "sum"), days_count=("date", "nunique"))
)
monthly_sales["days_in_month"] = monthly_sales["month_period"].dt.days_in_month
monthly_sales["is_complete_month"] = monthly_sales["days_count"] == monthly_sales["days_in_month"]

incomplete_months = monthly_sales.loc[~monthly_sales["is_complete_month"], "month_period"].dt.strftime("%Y-%m").tolist()
if incomplete_months:
    print("Incomplete months in monthly aggregation:", incomplete_months)
    print("Use caution when comparing monthly totals for incomplete months.")


fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_sales["month_period"], monthly_sales["sales_clean"], marker="o", label="monthly sales")
ax.set_title("Monthly Sales")
ax.set_xlabel("month")
ax.set_ylabel("sales_clean")
ax.legend()
ax.grid(True)
save_plot(IMAGES / "monthly_sales.png")
plt.show()

display(monthly_sales.tail())

## 5. Day-of-week pattern

In [ ]:
dow_sales = (
    df.groupby("day_of_week", as_index=False)
    .agg(avg_sales=("sales_clean", "mean"), median_sales=("sales_clean", "median"))
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(dow_sales["day_of_week"], dow_sales["avg_sales"], label="average sales")
ax.set_title("Average Sales by Day of Week")
ax.set_xlabel("day_of_week, Monday=0")
ax.set_ylabel("average sales_clean")
ax.legend()
ax.grid(True)
save_plot(IMAGES / "day_of_week_pattern.png")
plt.show()

display(dow_sales)

## 6. Promotion effect

In [ ]:
df["has_promo"] = df["onpromotion"] > 0
promo_summary = (
    df.groupby("has_promo", as_index=False)
    .agg(days=("date", "count"), avg_sales=("sales_clean", "mean"), median_sales=("sales_clean", "median"))
)
display(promo_summary)

## 7. Holiday effect

In [ ]:
if "is_holiday" in df.columns:
    holiday_summary = (
        df.groupby("is_holiday", as_index=False)
        .agg(days=("date", "count"), avg_sales=("sales_clean", "mean"), median_sales=("sales_clean", "median"))
    )
    display(holiday_summary)
else:
    print("Column is_holiday is not available.")

## 8. Oil price and transactions overview

In [ ]:
if "dcoilwtico" in df.columns:
    oil_corr = df[["sales_clean", "dcoilwtico"]].corr().iloc[0, 1]
    print(f"Correlation sales_clean vs dcoilwtico: {oil_corr:.4f}")
else:
    print("Column dcoilwtico is not available.")

if "transactions" in df.columns:
    transactions_corr = df[["sales_clean", "transactions"]].corr().iloc[0, 1]
    print(f"Correlation sales_clean vs transactions: {transactions_corr:.4f}")
else:
    print("Column transactions is not available.")

## 9. EDA conclusions

После запуска ноутбука перенесите ключевые наблюдения в `reports/01_eda_report.md`:

- есть ли выраженный тренд;
- есть ли недельный или месячный паттерн;
- отличаются ли продажи в промо-дни;
- заметно ли влияние праздников;
- насколько полезны внешние факторы для прогнозирования.